# Lab 03: API Testing - Test Clocks

**Lab**: 03-api-testing  
**Duration**: ~15 minutes  
**Prerequisites**: Completed `02_sandbox_and_test_cards.ipynb`

## Learning Objectives

By the end of this notebook, you will:
- Understand test clocks and their use cases
- Create and manage test clocks via the API
- Simulate subscription trials and renewals
- Test payment failures on subscription renewal

---

## Why Test Clocks?

Subscriptions operate over time. Without test clocks, testing a yearly subscription would require waiting an entire year!

### The Problem

```
Real Time Testing:
Day 1: Create subscription with 7-day trial
... wait 7 days ...
Day 8: Trial ends, first payment
... wait 30 days ...
Day 38: First renewal
... wait 365 days ...
Day 403: Annual renewal
```

### The Solution: Test Clocks

```
With Test Clocks:
Minute 1: Create subscription with trial
Minute 2: Advance clock 7 days -> trial ends
Minute 3: Advance clock 30 days -> renewal
Minute 4: Advance clock 1 year -> annual renewal
```

Test clocks let you simulate months or years of subscription activity in minutes!

In [ ]:
# Environment setup
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import stripe
import time

load_dotenv()
stripe.api_key = os.environ.get('STRIPE_SECRET_KEY')

# Verify connection
try:
    account = stripe.Account.retrieve()
    print(f"Connected to Stripe account: {account.id}")
    print(f"Mode: {'Sandbox' if 'test' in stripe.api_key else 'LIVE - BE CAREFUL!'}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key. Check your .env file.")

## How Test Clocks Work

### Key Concepts

1. **Frozen Time**: The simulated current time for the clock
2. **Customers**: Each test clock customer has their own time
3. **Subscriptions**: Billing events occur based on frozen time
4. **Webhooks**: Events fire as time advances

### Limitations

- Maximum **2 intervals** advance at a time
  - Monthly subscription -> max 2 months per advance
  - Yearly subscription -> max 2 years per advance
- Up to **3 customers** per test clock
- Test clocks work only in **sandboxes**

### Lifecycle

```
1. Create Clock (set frozen_time)
        |
        v
2. Create Customer (attached to clock)
        |
        v
3. Create Subscription
        |
        v
4. Advance Clock -> Events Fire
        |
        v
5. Repeat step 4 as needed
        |
        v
6. Delete Clock (cleanup)
```

## Section 1: Creating a Test Clock

Let's create a test clock and set its starting time.

<!-- PRESENTER: Explain that frozen_time is a Unix timestamp -->

In [ ]:
# Create a test clock
# Start time: Current time as Unix timestamp

frozen_time = int(time.time())

test_clock = stripe.test_helpers.TestClock.create(
    frozen_time=frozen_time,
    name="Workshop - Subscription Test"
)

print(f"Test Clock ID: {test_clock.id}")
print(f"Frozen Time: {datetime.fromtimestamp(test_clock.frozen_time)}")
print(f"Status: {test_clock.status}")

# Save for later use
CLOCK_ID = test_clock.id

### Checkpoint

You should see:
- A test clock ID starting with `clock_`
- The frozen time set to today
- Status: `ready`

**Dashboard**: Navigate to [Test Clocks](https://dashboard.stripe.com/test/test-clocks) to see your new clock.

## Section 2: Create Customer and Subscription

Now let's create a customer attached to this clock, then create a subscription.

### The Flow

1. Create a customer with `test_clock` parameter
2. Attach a test payment method
3. Create a subscription with a trial period

In [ ]:
# Create a customer attached to the test clock
customer = stripe.Customer.create(
    email="test-clock-demo@example.com",
    name="Test Clock Demo User",
    test_clock=CLOCK_ID,
    payment_method="pm_card_visa",
    invoice_settings={
        "default_payment_method": "pm_card_visa"
    }
)

print(f"Customer ID: {customer.id}")
print(f"Email: {customer.email}")
print(f"Test Clock: {customer.test_clock}")

CUSTOMER_ID = customer.id

In [ ]:
# Create a product and price for the subscription
product = stripe.Product.create(
    name="Workshop Pro Plan",
    description="Monthly subscription for test clock demo"
)

price = stripe.Price.create(
    product=product.id,
    unit_amount=2900,  # $29.00
    currency="usd",
    recurring={
        "interval": "month"
    }
)

print(f"Product: {product.name} ({product.id})")
print(f"Price: ${price.unit_amount / 100:.2f}/{price.recurring['interval']} ({price.id})")

PRICE_ID = price.id

In [ ]:
# Create a subscription with a 7-day free trial
subscription = stripe.Subscription.create(
    customer=CUSTOMER_ID,
    items=[{"price": PRICE_ID}],
    trial_period_days=7  # 7-day free trial
)

print(f"Subscription ID: {subscription.id}")
print(f"Status: {subscription.status}")
print(f"Trial End: {datetime.fromtimestamp(subscription.trial_end)}")

SUBSCRIPTION_ID = subscription.id

### Checkpoint

You should see:
- Subscription status: `trialing`
- Trial end: 7 days from the frozen time
- No payment charged yet (trial is free)

**Dashboard**: View the subscription at [Subscriptions](https://dashboard.stripe.com/test/subscriptions)

<!-- PRESENTER: Show the subscription details in Dashboard -->

## Section 3: Advancing Time

Now the powerful part - let's fast-forward time to trigger subscription events!

### What Happens When We Advance

| Advance To | Event |
|------------|-------|
| Trial end (day 7) | `customer.subscription.trial_will_end` (3 days before) |
| After trial | `customer.subscription.updated` (status: active) |
| After trial | `invoice.paid` (first payment) |
| End of month | `invoice.upcoming` |
| Start of next month | `invoice.paid` (renewal) |

In [ ]:
def advance_clock(clock_id: str, days: int):
    """Advance a test clock by a number of days."""
    # Get current frozen time
    clock = stripe.test_helpers.TestClock.retrieve(clock_id)
    current_time = clock.frozen_time
    
    # Calculate new time
    new_time = current_time + (days * 24 * 60 * 60)  # days to seconds
    
    print(f"Advancing clock from {datetime.fromtimestamp(current_time)}")
    print(f"                  to {datetime.fromtimestamp(new_time)}")
    print(f"                  ({days} days)")
    
    # Advance the clock
    advanced_clock = stripe.test_helpers.TestClock.advance(
        clock_id,
        frozen_time=new_time
    )
    
    # Wait for clock to finish advancing
    while advanced_clock.status == "advancing":
        time.sleep(1)
        advanced_clock = stripe.test_helpers.TestClock.retrieve(clock_id)
        print(f"  Status: {advanced_clock.status}...")
    
    print(f"Clock advanced! New time: {datetime.fromtimestamp(advanced_clock.frozen_time)}")
    return advanced_clock

print("Helper function defined!")

In [ ]:
# Advance 8 days to end the trial and trigger first payment
print("=" * 60)
print("Advancing past trial period...")
print("=" * 60)

advance_clock(CLOCK_ID, days=8)

# Check subscription status
subscription = stripe.Subscription.retrieve(SUBSCRIPTION_ID)
print(f"\nSubscription Status: {subscription.status}")
print(f"Trial End: {datetime.fromtimestamp(subscription.trial_end) if subscription.trial_end else 'None'}")

In [ ]:
# Check invoices for this customer
invoices = stripe.Invoice.list(customer=CUSTOMER_ID, limit=5)

print("Invoices:")
print("-" * 40)
for invoice in invoices.data:
    print(f"  {invoice.id}")
    print(f"    Amount: ${invoice.amount_paid / 100:.2f}")
    print(f"    Status: {invoice.status}")
    print(f"    Created: {datetime.fromtimestamp(invoice.created)}")
    print()

### Checkpoint

After advancing past the trial, you should see:
- Subscription status: `active` (no longer `trialing`)
- An invoice with status `paid` for $29.00
- The first payment was charged automatically

**Dashboard**: Check the [Invoices](https://dashboard.stripe.com/test/invoices) page.

In [ ]:
# Advance 30 more days to trigger a renewal
print("=" * 60)
print("Advancing to next billing cycle (renewal)...")
print("=" * 60)

advance_clock(CLOCK_ID, days=30)

# Check invoices again
invoices = stripe.Invoice.list(customer=CUSTOMER_ID, limit=5)

print("\nInvoices after renewal:")
print("-" * 40)
for invoice in invoices.data:
    print(f"  {invoice.id}: ${invoice.amount_paid / 100:.2f} - {invoice.status}")

## Section 4: Testing Payment Failure on Renewal

What happens when a customer's card fails at renewal? Let's simulate this.

### Scenario
1. Update customer's default payment method to a declining card
2. Advance to next billing cycle
3. Observe the failed payment and subscription status

In [ ]:
# Update customer to use a card that will decline

payment_method = stripe.PaymentMethod.attach(
  "pm_card_chargeCustomerFail",
  customer=CUSTOMER_ID,
)

stripe.Customer.modify(
    CUSTOMER_ID,
    invoice_settings={
        "default_payment_method": payment_method.id
    }
)

print("Customer updated to use declining card")
print("Next payment will fail!")

In [ ]:
# Advance to trigger renewal with failing card
print("=" * 60)
print("Advancing to trigger renewal (payment will fail)...")
print("=" * 60)

advance_clock(CLOCK_ID, days=30)

# Check subscription status
subscription = stripe.Subscription.retrieve(SUBSCRIPTION_ID)
print(f"\nSubscription Status: {subscription.status}")

# Check latest invoice
invoices = stripe.Invoice.list(customer=CUSTOMER_ID, limit=1)
if invoices.data:
    latest = invoices.data[0]
    print(f"\nLatest Invoice:")
    print(f"  ID: {latest.id}")
    print(f"  Amount Due: ${latest.amount_due / 100:.2f}")
    print(f"  Status: {latest.status}")
    print(f"  Attempt Count: {latest.attempt_count}")

### Checkpoint

After the failed renewal, you should see:
- Latest invoice status: `open` (not `paid`)
- Subscription status: `past_due` or still `active` (depends on retry settings)
- The invoice shows attempted payment that failed

This simulates what happens when a customer's card expires or has insufficient funds!

**Dashboard**: Check [Invoices](https://dashboard.stripe.com/test/invoices) to see the failed invoice.

## Cleanup

Test clocks and their associated objects persist until deleted. Let's clean up.

In [ ]:
# Clean up: Delete the test clock (also deletes associated customer/subscription)
print("Cleaning up test resources...")

try:
    # Cancel subscription first
    stripe.Subscription.cancel(SUBSCRIPTION_ID)
    print(f"  Cancelled subscription: {SUBSCRIPTION_ID}")
except Exception as e:
    print(f"  Could not cancel subscription: {e}")

try:
    # Delete test clock
    stripe.test_helpers.TestClock.delete(CLOCK_ID)
    print(f"  Deleted test clock: {CLOCK_ID}")
except Exception as e:
    print(f"  Could not delete clock: {e}")

print("\nCleanup complete!")

## Summary

In this notebook, you learned:

- **Test clocks** simulate time passage for subscriptions
- **Frozen time** is the starting point for your simulation
- **Advancing** the clock triggers billing events (trial end, renewal, etc.)
- **Payment failures** can be simulated by changing the payment method
- **Cleanup** is important - delete test clocks when done

## Lab Complete!

Congratulations! You've completed Lab 03: API Testing.

### What You Learned

- **Sandboxes**: Isolated test environments
- **Test Cards**: Simulate successful and failed payments
- **Test Clocks**: Simulate subscription lifecycle over time

### Next Steps

- Explore the [Stripe Testing Docs](https://docs.stripe.com/testing)
- Try the [Dashboard Test Clocks UI](https://dashboard.stripe.com/test/test-clocks)
- Review [Billing Testing Guide](https://docs.stripe.com/billing/testing)